<a href="https://colab.research.google.com/github/ramzyhafeel/SE4050-Deep-Learning-Assignment/blob/ResNet50-Dev/notebooks/03_resnet50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Member 3 — ResNet50 Transfer Learning

## Comparative Analysis of Deep Learning Architectures for Multi-Class Brain Tumour MRI Classification

**SE4050 – Deep Learning 2026**

This notebook implements a ResNet50 transfer-learning model for classifying brain tumour MRI images into four classes: **glioma**, **meningioma**, **notumor**, and **pituitary**.

### Architectural Context & Scientific Comparison
ResNet50 introduces **deep residual learning** via residual connections (identity shortcut mappings), addressing the vanishing gradient problem in very deep networks (He et al., 2016). Unlike VGG16 (a purely sequential deep network with standard feed-forward convolutional stacks), ResNet50 utilizes bottleneck residual blocks (`1x1` projection -> `3x3` conv -> `1x1` expansion) with skip connections and Batch Normalization.

---

### Important Scientific Limitations
1. **Transfer Learning Limitation:** ResNet50 starts from ImageNet pretrained weights, whereas Member 1's Custom CNN was trained from scratch. The comparison evaluates practical transfer-learning paradigms against custom from-scratch training, not an isolated architecture-only benchmark.
2. **Patient-Level Independence Limitation:** Patient-level independence could not be independently verified from the available dataset structure and identifiers. All analyses report image-level generalization.



---
## Phase 1 — Environment Verification and ResNet50 Preprocessing Pipeline

**Goal:** Verify environment reproducibility, validate Member 1's frozen split manifests via SHA-256 checksums, and construct the ResNet50-specific `tf.data` input pipeline.

### Phase 1 Tasks
1. Record complete runtime environment (Python, TensorFlow, Keras, NumPy, Pandas, scikit-learn, GPU, determinism).
2. Verify SHA-256 checksums for all three frozen split CSV files (`train.csv`, `val.csv`, `test.csv`).
3. Download exact Kaggle dataset version 1 (`masoudnickparvar/brain-tumor-mri-dataset/versions/1`).
4. Verify frozen sample counts: Train = 4,353, Validation = 1,089, Test = 1,311.
5. Verify frozen class mapping: `glioma: 0`, `meningioma: 1`, `notumor: 2`, `pituitary: 3`.
6. Verify that every referenced image file exists on disk.
7. Build ResNet50-specific `tf.data` pipeline using `keras.applications.resnet50.preprocess_input` (NO `Rescaling(1./255)`).
8. Apply common mild augmentation to training data only (rotation $\approx 0.03$, zoom $\approx \pm 0.08$, translation $\approx 0.05$).
9. Verify batch shapes, label integrity, and unshuffled validation stream.
10. Verify held-out test manifest without creating an evaluation dataset.
11. Save Phase 1 configuration and environment manifest.

### 1.1 — Install Dependencies and Imports

In [1]:
# Install kagglehub if not already present in environment
!pip install -q kagglehub

import os
import sys
import json
import hashlib
import time
import random
import platform
import shutil
import subprocess
import gc

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
import sklearn
import kagglehub
import matplotlib
import matplotlib.pyplot as plt

# Architecture-specific preprocessor for ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet50_preprocess_input

print("All imports completed successfully.")

All imports completed successfully.


### 1.2 — Set Random Seeds and Determinism

We fix the random seed to 42 across all libraries to ensure reproducible initialization, data shuffling, and pipeline execution. Deterministic op execution is requested where supported by hardware.

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    determinism_status = "TensorFlow op determinism enabled"
except Exception as e:
    determinism_status = f"Could not enable op determinism: {e}"

print(f"Seed:        {SEED}")
print(f"Determinism: {determinism_status}")

Seed:        42
Determinism: TensorFlow op determinism enabled


### 1.3 — Record Full Environment Information

In [3]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        gpu_name = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            text=True,
        ).strip().splitlines()[0]
        nvidia_smi = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'],
            text=True,
        ).strip().splitlines()[0]
    except Exception:
        gpu_name = tf.config.experimental.get_device_details(gpus[0]).get('device_name', 'Unknown GPU')
        nvidia_smi = 'Unavailable'
else:
    gpu_name = 'No GPU detected'
    nvidia_smi = 'N/A'

environment_info = {
    'member': 3,
    'model': 'ResNet50',
    'python_version': sys.version,
    'platform': platform.platform(),
    'tensorflow_version': tf.__version__,
    'keras_version': keras.__version__,
    'numpy_version': np.__version__,
    'pandas_version': pd.__version__,
    'scikit_learn_version': sklearn.__version__,
    'kagglehub_version': getattr(kagglehub, '__version__', 'unknown'),
    'matplotlib_version': matplotlib.__version__,
    'gpu_name': gpu_name,
    'nvidia_smi': nvidia_smi,
    'seed': SEED,
    'determinism_status': determinism_status,
}

print('Environment Recorded:')
for k, v in environment_info.items():
    print(f'  {k:<22}: {v}')

Environment Recorded:
  member                : 3
  model                 : ResNet50
  python_version        : 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
  platform              : Linux-6.6.122+-x86_64-with-glibc2.39
  tensorflow_version    : 2.20.0
  keras_version         : 3.13.2
  numpy_version         : 2.1.3
  pandas_version        : 2.2.3
  scikit_learn_version  : 1.6.1
  kagglehub_version     : 1.0.2
  matplotlib_version    : 3.10.0
  gpu_name              : No GPU detected
  nvidia_smi            : N/A
  seed                  : 42
  determinism_status    : TensorFlow op determinism enabled


### 1.4 — Download Pinned Kaggle Dataset (Version 1)

In strict compliance with the shared protocol, we download only the explicit pinned version 1 (`masoudnickparvar/brain-tumor-mri-dataset/versions/1`).

In [4]:
DATASET_HANDLE = "masoudnickparvar/brain-tumor-mri-dataset"
DATASET_VERSION = 1
PINNED_HANDLE = f"{DATASET_HANDLE}/versions/{DATASET_VERSION}"

dataset_path = kagglehub.dataset_download(PINNED_HANDLE)
print(f"Pinned Dataset Handle: {PINNED_HANDLE}")
print(f"Dataset downloaded to: {dataset_path}")
print(f"Dataset root contents: {os.listdir(dataset_path)}")

100%|██████████| 149M/149M [00:01<00:00, 106MB/s]

Extracting files...


Pinned Dataset Handle: masoudnickparvar/brain-tumor-mri-dataset/versions/1
Dataset downloaded to: /root/.cache/kagglehub/datasets/masoudnickparvar/brain-tumor-mri-dataset/versions/1
Dataset root contents: ['Testing', 'Training']


### 1.5 — Define Project Paths

In [7]:
from pathlib import Path
import os
import shutil
import subprocess

# ============================================================
# FRESH-COLAB PROJECT BOOTSTRAP
# ============================================================

REPO_URL = (
    "https://github.com/ramzyhafeel/"
    "SE4050-Deep-Learning-Assignment.git"
)

PROJECT_ROOT = Path(
    "/content/SE4050-Deep-Learning-Assignment"
)


def ensure_project_available():
    """
    Ensure the complete GitHub project exists
    in the current fresh Colab runtime.
    """

    required_items = [
        PROJECT_ROOT / "splits" / "train.csv",
        PROJECT_ROOT / "splits" / "val.csv",
        PROJECT_ROOT / "splits" / "test.csv",
        PROJECT_ROOT / "config",
        PROJECT_ROOT / "notebooks",
    ]

    if (
        PROJECT_ROOT.exists()
        and all(item.exists() for item in required_items)
    ):
        print("✓ Project repository already available.")
        return PROJECT_ROOT

    print("Project repository not found in this runtime.")
    print("Cloning GitHub repository...")

    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)

    subprocess.run(
        [
            "git",
            "clone",
            REPO_URL,
            str(PROJECT_ROOT),
        ],
        check=True,
    )

    missing = [
        str(item)
        for item in required_items
        if not item.exists()
    ]

    if missing:
        raise FileNotFoundError(
            "Repository cloned, but required project "
            f"files/folders are missing:\n{missing}"
        )

    print("✓ Repository cloned successfully.")
    return PROJECT_ROOT


# ============================================================
# INITIALIZE PROJECT PATHS
# ============================================================

PROJECT_ROOT = ensure_project_available()

# Alias in case later cells expect PROJECT_DIR
PROJECT_DIR = PROJECT_ROOT

SPLITS_DIR = PROJECT_ROOT / "splits"
CONFIG_DIR = PROJECT_ROOT / "config"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

RESULTS_DIR = PROJECT_ROOT / "results" / "resnet50"
MODELS_DIR = PROJECT_ROOT / "models" / "resnet50"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Split file variables used by later cells
TRAIN_CSV = SPLITS_DIR / "train.csv"
VAL_CSV = SPLITS_DIR / "val.csv"
TEST_CSV = SPLITS_DIR / "test.csv"

os.chdir(PROJECT_ROOT)


# ============================================================
# VERIFY PROJECT
# ============================================================

print("\n" + "=" * 70)
print("PROJECT SETUP")
print("=" * 70)

print("Project root :", PROJECT_ROOT)
print("Splits       :", SPLITS_DIR)
print("Config       :", CONFIG_DIR)
print("Notebooks    :", NOTEBOOKS_DIR)
print("Results      :", RESULTS_DIR)
print("Models       :", MODELS_DIR)

print("\nSplit files:")
print("train.csv    :", TRAIN_CSV)
print("val.csv      :", VAL_CSV)
print("test.csv     :", TEST_CSV)

assert TRAIN_CSV.is_file(), (
    f"train.csv missing: {TRAIN_CSV}"
)

assert VAL_CSV.is_file(), (
    f"val.csv missing: {VAL_CSV}"
)

assert TEST_CSV.is_file(), (
    f"test.csv missing: {TEST_CSV}"
)

assert CONFIG_DIR.exists(), (
    f"Config directory missing: {CONFIG_DIR}"
)

print("\nWorking directory:", Path.cwd())

print("\n✓ train.csv found")
print("✓ val.csv found")
print("✓ test.csv found")
print("✓ PROJECT READY")

Project repository not found in this runtime.
Cloning GitHub repository...
✓ Repository cloned successfully.

PROJECT SETUP
Project root : /content/SE4050-Deep-Learning-Assignment
Splits       : /content/SE4050-Deep-Learning-Assignment/splits
Config       : /content/SE4050-Deep-Learning-Assignment/config
Notebooks    : /content/SE4050-Deep-Learning-Assignment/notebooks
Results      : /content/SE4050-Deep-Learning-Assignment/results/resnet50
Models       : /content/SE4050-Deep-Learning-Assignment/models/resnet50

Split files:
train.csv    : /content/SE4050-Deep-Learning-Assignment/splits/train.csv
val.csv      : /content/SE4050-Deep-Learning-Assignment/splits/val.csv
test.csv     : /content/SE4050-Deep-Learning-Assignment/splits/test.csv

Working directory: /content/SE4050-Deep-Learning-Assignment

✓ train.csv found
✓ val.csv found
✓ test.csv found
✓ PROJECT READY


### 1.6 — Verify Frozen Split Checksums (SHA-256)

Before any model development, we verify the SHA-256 hashes of all three split CSV files against the frozen manifests created by Member 1. If any hash differs, execution is terminated immediately.

In [8]:
EXPECTED_CHECKSUMS = {
    "train.csv": "7273ef5fc2cb605c5b03b22a23cda0cb383ac0d9ce5949ac3c95649b5a4270cb",
    "val.csv":   "37b24456cfcd1b69df939e36603958eae6a9124b794c32c83a532b9018c6c88f",
    "test.csv":  "9afc40a38949eb4f46f8c9591d3b1f51cdd11aa991bfe2cf1d15c386c5537d39"
}

def compute_sha256(filepath):
    """Compute SHA-256 hash of a file."""
    hasher = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

checksums_verified = True
computed_checksums = {}

for filename, expected_hash in EXPECTED_CHECKSUMS.items():
    filepath = os.path.join(SPLITS_DIR, filename)
    if not os.path.exists(filepath):
        print(f"ERROR: {filepath} missing!")
        checksums_verified = False
        continue

    actual_hash = compute_sha256(filepath)
    computed_checksums[filename] = actual_hash
    matches = (actual_hash == expected_hash)
    if not matches:
        checksums_verified = False

    status = "MATCH" if matches else "MISMATCH"
    print(f"{filename:<10}: {status}")
    print(f"  Expected: {expected_hash}")
    print(f"  Computed: {actual_hash}")

if not checksums_verified:
    raise RuntimeError("STOP: Split manifest checksum mismatch! Do not proceed.")
else:
    print("\nAll three split checksums verified successfully.")

train.csv : MATCH
  Expected: 7273ef5fc2cb605c5b03b22a23cda0cb383ac0d9ce5949ac3c95649b5a4270cb
  Computed: 7273ef5fc2cb605c5b03b22a23cda0cb383ac0d9ce5949ac3c95649b5a4270cb
val.csv   : MATCH
  Expected: 37b24456cfcd1b69df939e36603958eae6a9124b794c32c83a532b9018c6c88f
  Computed: 37b24456cfcd1b69df939e36603958eae6a9124b794c32c83a532b9018c6c88f
test.csv  : MATCH
  Expected: 9afc40a38949eb4f46f8c9591d3b1f51cdd11aa991bfe2cf1d15c386c5537d39
  Computed: 9afc40a38949eb4f46f8c9591d3b1f51cdd11aa991bfe2cf1d15c386c5537d39

All three split checksums verified successfully.


### 1.7 — Load Split CSVs and Verify Sample Counts

In [9]:
train_df = pd.read_csv(os.path.join(SPLITS_DIR, "train.csv"))
val_df   = pd.read_csv(os.path.join(SPLITS_DIR, "val.csv"))
test_df  = pd.read_csv(os.path.join(SPLITS_DIR, "test.csv"))

EXPECTED_TRAIN = 4353
EXPECTED_VAL   = 1089
EXPECTED_TEST  = 1311

print(f"Train samples: {len(train_df):<5} (expected {EXPECTED_TRAIN})")
print(f"Val samples:   {len(val_df):<5} (expected {EXPECTED_VAL})")
print(f"Test samples:  {len(test_df):<5} (expected {EXPECTED_TEST})")

assert len(train_df) == EXPECTED_TRAIN, f"Train count mismatch: {len(train_df)}"
assert len(val_df)   == EXPECTED_VAL,   f"Val count mismatch: {len(val_df)}"
assert len(test_df)  == EXPECTED_TEST,  f"Test count mismatch: {len(test_df)}"
print("\nAll sample counts match expected frozen split sizes.")

Train samples: 4353  (expected 4353)
Val samples:   1089  (expected 1089)
Test samples:  1311  (expected 1311)

All sample counts match expected frozen split sizes.


### 1.8 — Verify Class Names and Index Mapping

The class mapping must remain strictly identical across all models:
`glioma: 0`, `meningioma: 1`, `notumor: 2`, `pituitary: 3`.

In [10]:
CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
CLASS_TO_INDEX = {name: idx for idx, name in enumerate(CLASS_NAMES)}

with open(os.path.join(CONFIG_DIR, "class_to_index.json"), "r") as f:
    frozen_class_mapping = json.load(f)

print("Class Mapping Verification:")
for name, idx in frozen_class_mapping.items():
    print(f"  {name:<12} -> {idx}")

assert CLASS_TO_INDEX == frozen_class_mapping, "Class mapping mismatch!"

# Verify class distributions in splits
for split_name, df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"\n{split_name} Distribution:")
    dist = df["label"].value_counts().sort_index()
    for cls_name in CLASS_NAMES:
        print(f"  {cls_name:<12}: {dist.get(cls_name, 0)}")

Class Mapping Verification:
  glioma       -> 0
  meningioma   -> 1
  notumor      -> 2
  pituitary    -> 3

Train Distribution:
  glioma      : 1057
  meningioma  : 1064
  notumor     : 1080
  pituitary   : 1152

Validation Distribution:
  glioma      : 264
  meningioma  : 267
  notumor     : 270
  pituitary   : 288

Test Distribution:
  glioma      : 300
  meningioma  : 306
  notumor     : 405
  pituitary   : 300


### 1.9 — Verify That Every Referenced Image Exists on Disk

In [11]:
missing_images = []
for split_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    for _, row in df.iterrows():
        full_path = os.path.join(dataset_path, row["filepath"])
        if not os.path.isfile(full_path):
            missing_images.append((split_name, row["filepath"]))

if missing_images:
    print(f"ERROR: {len(missing_images)} images missing!")
    raise RuntimeError("Missing image files detected.")
else:
    total_images = len(train_df) + len(val_df) + len(test_df)
    print(f"All {total_images} referenced images verified on disk.")

All 6753 referenced images verified on disk.


---
## ResNet50-Specific Preprocessing Pipeline

### Why `keras.applications.resnet50.preprocess_input` instead of `Rescaling(1./255)`?

Pretrained ResNet50 in Keras was trained with the standard Caffe-style preprocessing pipeline:
1. Converts RGB to BGR.
2. Zero-centers each color channel with respect to ImageNet channel means: $[103.939, 116.779, 123.68]$.
3. Operates on raw float pixel values in $[0, 255]$.

**Critical Rule:** Applying `Rescaling(1./255)` before `resnet50.preprocess_input` would result in **inappropriate double normalization**, scaling images to $[0, 1]$ and then subtracting $\approx 100$, producing severely corrupted activations. The image decoder therefore outputs float32 values in $[0, 255]$, and `resnet50.preprocess_input` handles the transformation.

### 1.10 — Define Pipeline Constants

In [12]:
IMAGE_HEIGHT = 224
IMAGE_WIDTH  = 224
CHANNELS     = 3
IMAGE_SIZE   = (IMAGE_HEIGHT, IMAGE_WIDTH)
INPUT_SHAPE  = (IMAGE_HEIGHT, IMAGE_WIDTH, CHANNELS)
BATCH_SIZE   = 16
NUM_CLASSES  = 4
MAX_EPOCHS   = 20

print(f"Image Shape:  {INPUT_SHAPE}")
print(f"Batch Size:   {BATCH_SIZE}")
print(f"Num Classes:  {NUM_CLASSES}")
print(f"Max Epochs:   {MAX_EPOCHS}")

Image Shape:  (224, 224, 3)
Batch Size:   16
Num Classes:  4
Max Epochs:   20


### 1.11 — Define Training-Only Augmentation Layer

We apply the exact same mild, conservative augmentation policy used by Custom CNN and VGG16:
- Random Rotation: factor = 0.03 ($\approx \pm 10.8^\circ$)
- Random Zoom: $\pm 8\%$
- Random Translation: $5\%$
Applied **strictly to training data**.

In [13]:
def make_augmentation(seed=SEED):
    return tf.keras.Sequential([
        tf.keras.layers.RandomRotation(factor=0.03, seed=seed),
        tf.keras.layers.RandomZoom(
            height_factor=(-0.08, 0.08),
            width_factor=(-0.08, 0.08),
            seed=seed,
        ),
        tf.keras.layers.RandomTranslation(
            height_factor=0.05,
            width_factor=0.05,
            seed=seed,
        ),
    ], name='augmentation')

# Preview/smoke instance. Each controlled experiment gets a fresh instance.
augmentation_layer = make_augmentation(SEED)
print('Training-only augmentation factory defined.')

Training-only augmentation factory defined.


### 1.12 — Build `tf.data` Pipeline with ResNet50 Preprocessing

In [14]:
def decode_and_resize(filepath, label):
    """Decode JPEG, resize to 224x224 and keep float32 values in [0,255]."""
    raw = tf.io.read_file(filepath)
    image = tf.image.decode_jpeg(raw, channels=CHANNELS)
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32)
    return image, label

def build_dataset(df, is_training=False):
    """Build raw-image tf.data pipeline. Model owns augmentation + preprocessing."""
    filepaths = [os.path.join(str(dataset_path), fp) for fp in df['filepath'].values]
    labels = [CLASS_TO_INDEX[lbl] for lbl in df['label'].values]
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if is_training:
        ds = ds.shuffle(
            buffer_size=len(filepaths),
            seed=SEED,
            reshuffle_each_iteration=True,
        )
    ds = ds.map(decode_and_resize, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE, drop_remainder=False)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

print('ResNet50 dataset builder defined.')
print('tf.data keeps pixels in raw [0,255] range; augmentation and ResNet50 preprocess_input run exactly once inside the model.')

ResNet50 dataset builder defined.
tf.data keeps pixels in raw [0,255] range; augmentation and ResNet50 preprocess_input run exactly once inside the model.


### 1.13 — Instantiate Training and Validation Datasets

In [15]:
train_ds = build_dataset(train_df, is_training=True)
val_ds   = build_dataset(val_df,   is_training=False)

train_batches = tf.data.experimental.cardinality(train_ds).numpy()
val_batches   = tf.data.experimental.cardinality(val_ds).numpy()

print(f"Training Batches:   {train_batches} ({len(train_df)} images)")
print(f"Validation Batches: {val_batches} ({len(val_df)} images)")

Training Batches:   273 (4353 images)
Validation Batches: 69 (1089 images)
